# Занятие 10 (дополнительно): смешанные модели для бинарных данных — GLMM
### Рабочая тетрадь студента
*Курс: Байесовский анализ эмпирических данных (2026)*

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic_ru.ipynb)
*English version: [10_mixed_models_logistic.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic.ipynb)*

На занятии 10 мы строили одну бинарную модель на наблюдение. Реальные бинарные данные часто *продольные*: один пациент отвечает на нескольких визитах, один избиратель наблюдается на нескольких волнах. Это приложение к странице «Классические смешанные модели и байесовская иерархия» добавляет аппарат случайных эффектов к логистической связи на двух датасетах: классическое **исследование лечения депрессии** из книги Агрести *Categorical Data Analysis* (340 пациентов, три визита) и данные голосования **ANES 2016**. Мы пройдём ту же лестницу, что и в линейной тетради занятия 9 — плоская модель, наивная плоская модель, классический GLMM, байесовский иерархический GLMM, — и позволим PSIS-LOO рассудить.


In [ ]:
import sys, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

try:
    import arviz as az
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arviz>=0.14.0"])
    import arviz as az

try:
    import bambi as bmb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bambi>=0.13.0"])
    import bambi as bmb

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def fetch(fname, url):
    # Canonical copy in the book repository; auto-download in Colab or when missing.
    path = os.path.join("data", fname)
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(path):
        import urllib.request
        urllib.request.urlretrieve(url, path)
    return path

BASE = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/"
DEPR_PATH = fetch("Depression.dat", BASE + "Depression.dat")
ANES_PATH = fetch("anes_clinton.csv", BASE + "anes_clinton.csv")

print("Environment ready: statsmodels", sm.__version__, "| Bambi", bmb.__version__, "| ArviZ", az.__version__)


## 1. Продольный бинарный исход: исследование лечения депрессии

### Идея
Исследование (распространяется как таблица 9.1 в материалах книги Агрести *Categorical Data Analysis*) — продольное клиническое испытание, сравнивающее **новый препарат** со **стандартным** у пациентов с депрессией. Пациентов классифицировали по исходной тяжести — *лёгкая* или *тяжёлая* — случайно назначали один из двух препаратов и классифицировали состояние как *норма* (1) или *патология* (0) после 1, 2 и 4 недель лечения.

Важная деталь моделирования: когда шкала времени отражает **накопленную дозу**, логит-модель часто ведёт себя приблизительно линейно по **логарифму времени**. Недели 1, 2, 4 дают аккуратные баллы $t = \log_2(\text{неделя}) = 0, 1, 2$. Агрести отмечает, что с номерами недель получаются похожие содержательные результаты; лог-баллы — теоретически мотивированный выбор по умолчанию.

### Пример


In [ ]:
data_depr = pd.read_csv(DEPR_PATH, sep=r"\s+")
print(data_depr.head())
print("\nShape:", data_depr.shape,
      "| cases:", data_depr["case"].nunique(),
      "| rows per case:", sorted(data_depr.groupby("case").size().unique()))
print("severity: 0 = mild, 1 = severe | drug: 0 = standard, 1 = new | time: log2(week) in {0,1,2}")

# Descriptive proportion of 'normal' responses by cell
agg = data_depr.groupby(["severity", "drug", "time"])["outcome"].mean().reset_index()
print("\nShare of normal outcomes by cell:")
print(agg.pivot(index=["severity", "drug"], columns="time", values="outcome").round(3))

sns.catplot(x="time", y="outcome", hue="drug", col="severity", data=agg, kind="bar",
            palette="rocket", legend_out=False, height=3.6, aspect=1.1)
plt.suptitle("Proportion normal by week, drug and baseline severity", y=1.05)
plt.show()


### На что обратить внимание
Читаем четыре панели как факторную историю $2 \times 2$:
- Обе группы начинают около уровня своей исходной тяжести (лёгкая $\approx 0{,}5$, тяжёлая $\approx 0{,}2$).
- На **стандартном** препарате улучшение медленное и скромное в обеих группах тяжести.
- На **новом** препарате кривые взлетают: к 4-й неделе (t=2) 97% исходно лёгких и 83% исходно тяжёлых пациентов классифицированы как норма.
- Разрыв между препаратами *растёт со временем* — первый намёк на **взаимодействие препарат × время** на логит-шкале, которое формализуют плоские модели ниже.

И структурный момент: один и тот же `case` встречается трижды. Эти 1020 строк — не 1020 независимых людей; это 340 человек, измеренных трижды.


## 2. Плоские логистические модели: наивные базовые уровни

### Идея
Сделаем вид, что все строки независимы, и подгоним обычные логистические регрессии — сначала только с `drug`, затем с полной фиксированной структурой `drug + time + severity + drug:time`. Это ровно крайность «полного объединения» из занятия 9, теперь на бинарном исходе. Сравнение двух подгонок уже рассказывает содержательную историю; графики остатков показывают, почему «остатки» бинарных данных нужно читать осторожно.

### Пример


In [ ]:
basic1 = smf.logit(formula="outcome ~ drug", data=data_depr).fit(disp=0)
print(basic1.summary())
print("\nOdds ratio for drug:", round(float(np.exp(basic1.params["drug"])), 2))


In [ ]:
basic2 = smf.logit(formula="outcome ~ drug + time + severity + drug:time", data=data_depr).fit(disp=0)
print(basic2.summary())

# Binned residuals: for binary y, raw residuals always form two bands — bin them first.
def binned_residuals(res, data, x="fitted", n_bins=10, ax=None):
    pred = res.fittedvalues if x == "fitted" else data[x]
    df = pd.DataFrame({"x": pred, "r": res.resid_generalized})
    if df["x"].nunique() <= n_bins:
        df["bin"] = df["x"]          # discrete designs (e.g. drug-only): bin by fitted value
    else:
        df["bin"] = pd.qcut(df["x"], n_bins)
    binned = df.groupby("bin", observed=True).mean(numeric_only=True)
    ax.plot(binned["x"], binned["r"], "o-", color="C3")
    ax.axhline(0, color="gray", lw=0.8)
    return binned

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(basic1.fittedvalues, basic1.resid_generalized, s=8, alpha=0.5, color="C0")
axs[0].set_title("drug-only: raw residuals vs fitted")
binned_residuals(basic1, data_depr, ax=axs[1])
axs[1].set_title("drug-only: binned mean residuals")
for ax in axs: ax.set_xlabel("fitted probability")
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(basic2.fittedvalues, basic2.resid_generalized, s=8, alpha=0.5, color="C0")
axs[0].set_title("full flat model: raw residuals vs fitted")
binned_residuals(basic2, data_depr, ax=axs[1])
axs[1].set_title("full flat model: binned mean residuals")
for ax in axs: ax.set_xlabel("fitted probability")
plt.tight_layout(); plt.show()


### На что обратить внимание
1. В модели только с препаратом новый препарат выглядит равномерно лучше ($\widehat{\operatorname{OR}} \approx 2$). В полной модели **главный эффект `drug` схлопывается до $\approx -0{,}06$**, а `drug:time` взлетает до $\approx +1{,}02$: препараты похожи на старте и расходятся с накопленной дозой. Главный эффект без своего взаимодействия — это *средневзвешенное по времени*, бессмысленная сводка, когда взаимодействие реально.
2. Панели сырых остатков — две полосы по построению ($y - p$ при $y \in \{0,1\}$); они не несут диагностического сообщения. Полезная версия — *бинованные* средние остатки: для модели только с препаратом они систематически изгибаются — структуры не хватает. (Бинованные графики остатков — бинарный аналог проверок residual-vs-fitted по Гельману и Хиллу.)
3. Каждое число здесь вычислено при фикции независимых строк, поэтому все стандартные ошибки занижены — плата за игнорирование `case`.


## 3. Классическая попытка GLMM: `BinomialBayesMixedGLM`

### Идея
Добавим случайный интерсепт на пациента: `outcome ~ drug + time + severity + drug:time + (1 | case)`. Точное правдоподобие для логистического GLMM требует численного интегрирования по 340 эффектам пациентов, что классический софт обходит приближениями. `statsmodels` предлагает *вариационное байесовское* приближение (`fit_vb`): оно возвращает приближённые апостериорные средние и стандартные отклонения фиксированных эффектов, но **без сэмплов апостериорного распределения, без диагностик сходимости, без полной апостериорной оценки компоненты дисперсии** — гауссово приближение вокруг моды, а не настоящая вещь.

### Пример


In [ ]:
random_effects_spec = {"a": "0 + C(case)"}
mixed_vb = BinomialBayesMixedGLM.from_formula(
    "outcome ~ drug + time + severity + drug:time", random_effects_spec, data_depr)
result_vb = mixed_vb.fit_vb()
print(result_vb.summary())

fe = pd.DataFrame({"VB mean": result_vb.fe_mean,
                   "VB sd": result_vb.fe_sd,
                   "flat logit": basic2.params.values},
                  index=["Intercept", "drug", "time", "severity", "drug:time"])
print("\nFixed effects: VB mixed GLMM vs flat logit MLE")
print(fe.round(3))

re_vb = result_vb.random_effects()
print("\nFirst patient effects (VB):")
print(re_vb.head().round(3))
print("Estimated SD of case effects (VB):", round(float(result_vb.vc_mean[0]), 3))


### На что обратить внимание
VB-оценки попадают близко к плоской MLE — история фиксированных эффектов устойчива. Чего таблица дать не может:
- апостериорного распределения для стандартного отклонения на уровне пациентов (только одно число `vc_mean`);
- предсказательного распределения для *нового* пациента;
- способа проверить качество вариационного приближения (ELBO вырос, но насколько он ошибается?).

Прежняя тетрадь говорила прямо: с одними точечными оценками «сложно оценить качество и делать количественные выводы — насколько, какой размер эффекта, какая неопределённость?» Байесовская иерархическая модель ниже отвечает ровно на эти вопросы.


## 4. Байесовский иерархический GLMM в Bambi и PSIS-LOO

### Идея
Та же формула, теперь с полным апостериорным распределением: каждый пациент получает интерсепт $b_{\text{case}}$, извлечённый из $\operatorname{Normal}(0, \sigma_{\text{case}})$, а $\sigma_{\text{case}}$ — сам параметр с апостериорным распределением. Подгоним три вложенные модели и позволим PSIS-LOO (критерий курса; прежняя тетрадь печатала WAIC, которого в ArviZ 1.x больше нет) ранжировать их по вневыборочной предсказательной точности:
1. только `drug` — наивный базовый уровень с полным объединением;
2. `drug + time + severity + drug:time` — полная плоская модель;
3. то же плюс `(1 | case)` — иерархический GLMM.

### Пример


In [ ]:
FIXED = "drug + time + severity + drug:time"

model1 = bmb.Model("outcome ~ drug", data_depr, family="bernoulli")
idata1 = model1.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})
print(az.summary(idata1, round_to=2))


In [ ]:
model2 = bmb.Model(f"outcome ~ {FIXED}", data_depr, family="bernoulli")
idata2 = model2.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})
print(az.summary(idata2, round_to=2))
print("\nPSIS-LOO (ELPD):", az.loo(idata2).elpd.round(1))


In [ ]:
model3 = bmb.Model(f"outcome ~ {FIXED} + (1 | case)", data_depr, family="bernoulli")
print(model3)
idata3 = model3.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    target_accept=0.95, idata_kwargs={"log_likelihood": True})
print(az.summary(idata3, var_names=["Intercept", "drug", "time", "severity",
                                    "drug:time", "1|case_sigma"], round_to=2))
print("\nPSIS-LOO (ELPD):", az.loo(idata3).elpd.round(1))


In [ ]:
cmp_df = az.compare({"M1: drug only (flat)": idata1,
                        "M2: full fixed (flat)": idata2,
                        "M3: + (1|case) GLMM": idata3})
print(cmp_df.round(1))
az.plot_compare(cmp_df)
plt.show()


### На что обратить внимание
1. **Сначала проверки доверия**: `r_hat` $\le 1{,}01$ и здоровый ESS в сводках выше. Даже при `target_accept=0.95` GLMM сообщает о нескольких дивергенциях: при всего трёх бинарных наблюдениях на пациента $\sigma_{\text{case}}$ слабо идентифицируем, его апостериорное распределение скапливается у нуля, и сэмплер чувствует геометрию воронки из занятия 6 — та же приграничная территория, что и «сингулярная подгонка» `lme4` со страницы книги.
2. **Вердикт LOO — ничья**: полная плоская модель и GLMM различаются примерно на один ELPD (dse $< 1$). С тремя наблюдениями на пациента данные не могут отличить «нет гетерогенности пациентов» от «небольшой гетерогенности»: частичное объединение вырождается в полное, и вневыборочному предсказанию всё равно. Сравните со `sleepstudy` из тетради занятия 9, где десять непрерывных наблюдений на испытуемого позволили иерархии выиграть уверенно. **Иерархия окупается, когда кластеры информативны, — а не просто потому, что строки повторяются.**
3. Апостериорное распределение `1|case_sigma` (среднее $\approx 0{,}3$, масса у нуля) — ровно тот ответ, на который VB-таблица могла лишь намекнуть: полное утверждение о неопределённости межпациентской гетерогенности, недоступное классическому фиту, — и здесь оно говорит, что гетерогенность мала.
4. Предупреждения Парето-$k$ выше $0{,}7$ обычны для иерархических моделей (каждая строка опирается на эффект своего пациента); считайте их поводом рассмотреть кросс-валидацию с исключением целого *пациента*, а не приговором.

Следующая ячейка отчитывается там, где это важно, — на **наблюдаемой шкале вероятностей**, с названными базовыми уровнями (протокол курса: контрасты при указанных базовых значениях, никогда не голые лог-одды):


In [ ]:
def cell_probability(idata, severity, drug, t, var_names):
    # Posterior of P(normal) for a TYPICAL patient (case effect = 0)
    post = idata.posterior.to_dataset()
    eta = (post["Intercept"] + drug * post["drug"] + t * post["time"]
           + severity * post["severity"] + drug * t * post["drug:time"])
    return 1 / (1 + np.exp(-eta)).values.flatten()

rows = []
for sev, sev_name in [(0, "mild"), (1, "severe")]:
    for dr, dr_name in [(0, "standard"), (1, "new")]:
        for t, week in [(0, 1), (1, 2), (2, 4)]:
            p = cell_probability(idata3, sev, dr, t, None)
            lo, hi = az.hdi(p, prob=0.94)
            rows.append({"severity": sev_name, "drug": dr_name, "week": week,
                         "P(normal) mean": p.mean().round(3),
                         "94% HDI": f"[{lo:.2f}, {hi:.2f}]"})
report = pd.DataFrame(rows)
print("M3 (hierarchical GLMM), typical patient, posterior P(normal):")
print(report.to_string(index=False))

# Probability-scale drug effect at week 4 for each severity
for sev, sev_name in [(0, "mild"), (1, "severe")]:
    p_new = cell_probability(idata3, sev, 1, 2, None)
    p_std = cell_probability(idata3, sev, 0, 2, None)
    delta = p_new - p_std
    lo, hi = az.hdi(delta, prob=0.94)
    print(f"week 4, {sev_name}: P(new) - P(standard) = {delta.mean():+.3f}, 94% HDI [{lo:+.2f}, {hi:+.2f}]")


## 5. Поперечные взаимодействия: голосование ANES 2016

### Идея
Не все бинарные данные продольные, но тот же аппарат обрабатывает и *неоднородные группы* в поперечных срезах. Пример ANES 2016 (из документации Bambi) моделирует голосование Клинтон/Трамп с **партийно-специфичными наклонами возраста**. Одна тонкость кодирования: формула `vote['clinton'] ~ party_id + party_id:age` разворачивает взаимодействие в наклон возраста **для каждой партии** (три наклона: демократы, независимые, республиканцы), так что модель полного ранга — главный эффект возраста поглощён партийными наклонами, а не выброшен. `vote['clinton']` сообщает Bambi, какой уровень отклика является моделируемым «успехом».

### Пример


In [ ]:
clinton_data = pd.read_csv(ANES_PATH)
print(clinton_data.head())
print("\nRows:", len(clinton_data), "| votes:", clinton_data["vote"].value_counts().to_dict(),
      "| parties:", sorted(clinton_data["party_id"].unique()))

clinton_model = bmb.Model("vote['clinton'] ~ party_id + party_id:age", clinton_data,
                          family="bernoulli")
print(clinton_model)
idata_anes = clinton_model.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                               target_accept=0.85, idata_kwargs={"log_likelihood": True})
print(az.summary(idata_anes, round_to=2))


In [ ]:
# Report the age effects on the observable scale: probability change per decade of age,
# evaluated at each party's own baseline (mean age within party).
post = idata_anes.posterior.to_dataset()
party_levels = ["democrat", "independent", "republican"]
base_age = {p: clinton_data.loc[clinton_data["party_id"] == p, "age"].mean()
            for p in party_levels}
slopes = {p: post["party_id:age"].sel(**{"party_id:age_dim": p}).values.flatten()
          for p in party_levels}

fig, ax = plt.subplots(figsize=(7, 3.8))
for i, p in enumerate(party_levels):
    ax.hist(slopes[p], bins=60, alpha=0.55, density=True, label=p, color=f"C{i}")
ax.set_xlabel("posterior log-odds slope per year of age")
ax.set_title("Party-specific age slopes (ANES 2016, Clinton vote)")
ax.legend()
plt.tight_layout(); plt.show()

print("Per-decade effect on P(vote Clinton), at each party's mean age (divide-by-4 rule as a sanity check):")
for p in party_levels:
    s10 = slopes[p] * 10
    eta = post["Intercept"].values.flatten()
    if p != "democrat":   # 'democrat' is the reference level, absorbed into the Intercept
        eta = eta + post["party_id"].sel(**{"party_id_dim": p}).values.flatten()
    p_hat = 1 / (1 + np.exp(-eta))
    delta_prob = s10 * p_hat * (1 - p_hat)   # logit-approximation of the probability change
    lo, hi = az.hdi(delta_prob, prob=0.94)
    print(f"  {p:12s} baseline age {base_age[p]:.0f}: "
          f"DeltaP/decade = {delta_prob.mean():+.3f}, 94% HDI [{lo:+.3f}, {hi:+.3f}]")


### На что обратить внимание
1. Три апостериорных распределения наклонов почти не перекрываются: возраст *негативно* предсказывает голос за Клинтон у республиканцев и независимых, а у демократов наклон близок к нулю. «Средний эффект возраста» скрыл бы всю историю — взаимодействия делают гетерогенность видимой.
2. Мы перевели наклоны на шкалу вероятностей **при собственной базовой вероятности каждой партии** (золотое правило курса из занятия 10): один и тот же наклон логита означает разные изменения вероятности для демократов ($p \approx 0{,}9$) и республиканцев ($p \approx 0{,}1$).
3. Прежняя тетрадь использовала `draws=200, target_accept=0.85`; мы подняли число сэмплов до 1000 и сохранили `target_accept=0.85` — всегда проверяйте колонку r_hat, прежде чем цитировать числа.


## 6. Необязательное домашнее задание: просоциальные шимпанзе как GLMM

Прежняя тетрадь ссылалась на «примеры GLM про морские звёзды и шимпанзе» из других материалов курса. Указатель на морских звёзд не имел присоединённых данных; шимпанзе уже есть в этой книге как капстоун занятия 10 — поэтому домашнее задание делает связь явной.

**Задания:**
1. Загрузите данные шимпанзе из репозитория книги (`data/chimpanzees.csv`, см. тетрадь занятия 10) и вспомните дизайн: `pulled_left`, `prosoc_left`, `condition`, `actor`, `block`.
2. Подгоните плоскую логистическую модель `pulled_left ~ prosoc_left * condition`, игнорируя акторов и блоки.
3. Подгоните GLMM `pulled_left ~ prosoc_left * condition + (1 | actor) + (1 | block)` в Bambi.
4. Сравните обе модели по PSIS-LOO. Отчитайтесь о популяционном эффекте просоциальной опции на P(потянуть левый рычаг) с неопределённостью и об одном контрасте на уровне акторов по вашему выбору.
5. Одно предложение о том, что даёт `(1 | block)` — и проверьте, поддерживает ли таблица LOO его включение.


In [ ]:
# Homework data starter (no solutions below — this one is yours to build)
CHIMP_PATH = fetch("chimpanzees.csv", BASE + "chimpanzees.csv")
chimp = pd.read_csv(CHIMP_PATH, sep=";")
print(chimp.head())
print("actors:", chimp["actor"].nunique(), "| blocks:", chimp["block"].nunique(),
      "| trials:", len(chimp))


## Итоги

- Продольные бинарные данные имеют ту же кластерную структуру, что и продольные непрерывные; лечение то же — случайные эффекты, — только вокруг другой функции связи.
- Плоская логистическая лестница уже несёт содержательную историю (главный эффект препарата схлопывается, как только входит взаимодействие со временем), но её стандартные ошибки — фикция.
- Вариационный GLMM из `statsmodels` даёт точечные оценки и приближённые SD; байесовский иерархический GLMM даёт полное апостериорное распределение — включая SD между пациентами — и арбитраж PSIS-LOO между плоской и иерархической структурами. В этом испытании арбитраж назвал ничью (три бинарные строки на пациента несут слишком мало кластерной информации); на `sleepstudy` иерархия выиграла уверенно.
- Отчётность принадлежит шкале вероятностей при названных базовых уровнях: «к 4-й неделе у исходно лёгких пациентов: +X процентных пунктов на новом препарате, 94% HDI [a, b]».
- Литература: Agresti, *Categorical Data Analysis*, пример с клиническим испытанием лечения депрессии; Gelman & Hill, *Data Analysis Using Regression and Multilevel/Hierarchical Models*, бинованные остатки и логистические модели со случайным интерсептом; документация Bambi для случая ANES.

**Назад к книге:** капстоун занятия 10 (просоциальные шимпанзе) строит иерархический логит вручную; страница «Классические смешанные модели и байесовская иерархия» сжимает эту лестницу; занятие 11 добавляет полный диагностический арсенал.
